# DNA Sequence Classification using Machine Learning & Deep Learning


In [ ]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg
import pickle
import joblib

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
human = pd.read_table('human_data.txt')
human.head()


In [ ]:
human.info()


In [ ]:
img = mpimg.imread("gene family.png")
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Gene Family Mapping")
plt.show()


### Apply k-mer counting to convert DNA sequences into words


In [ ]:
# Function to convert sequence strings into k-mer words, default size = 6 (hexamer words)
def getKmers(sequence, size=6):
    return [sequence[x:x+size].lower() for x in range(len(sequence) - size + 1)]


In [ ]:
human['words'] = human.apply(lambda x: getKmers(x['sequence']), axis=1)
human = human.drop('sequence', axis=1)
human.head()


### Convert lists of k-mers into string sentences


In [ ]:
human_texts = list(human['words'])
for item in range(len(human_texts)):
    human_texts[item] = ' '.join(human_texts[item])
y_data = human.iloc[:, 0].values

print('Sample k-mer sentence (first 80 chars):')
print(human_texts[0][:80], '...')
print('Class label:', y_data[0])


### Bag of Words with CountVectorizer


In [ ]:
# Create CountVectorizer with 4-gram of hexamers
cv = CountVectorizer(ngram_range=(4, 4))
X = cv.fit_transform(human_texts)
print('X shape (samples, features):', X.shape)


### Class Balance of Human Dataset


In [ ]:
plt.figure(figsize=(7, 4))
human['class'].value_counts().sort_index().plot.bar(color='steelblue')
plt.title('Class Balance in Human Dataset')
plt.xlabel('Gene Family Class')
plt.ylabel('Count')
plt.show()


### Model Evaluation Helper


In [ ]:
def get_metrics(y_test, y_predicted):
    accuracy = accuracy_score(y_test, y_predicted)
    precision = precision_score(y_test, y_predicted, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_predicted, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_predicted, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1


### 5-Fold Cross Validation for Machine Learning Models


In [ ]:
# Decision Tree
acc_dt_list, f1_dt_list = [], []
y = y_data.copy()
print('--- Decision Tree Classifier ---')
for i, (train_index, val_index) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y)):
    dt = DecisionTreeClassifier(random_state=42, max_depth=30)
    dt.fit(X[train_index], y[train_index])
    pred = dt.predict(X[val_index])
    acc, prec, rec, f1 = get_metrics(y[val_index], pred)
    print(f'Fold {i+1}: Accuracy = {acc:.4f} | F1 = {f1:.4f}')
    acc_dt_list.append(acc)
    f1_dt_list.append(f1)

acc_dt = np.mean(acc_dt_list)
f1s_dt = np.mean(f1_dt_list)
print(f'Mean CV Accuracy: {acc_dt*100:.2f}% ± {np.std(acc_dt_list)*100:.2f}% | Mean F1: {f1s_dt*100:.2f}%')


In [ ]:
# Random Forest
acc_rf_list, f1_rf_list = [], []
print('--- Random Forest Classifier ---')
for i, (train_index, val_index) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y)):
    rf = RandomForestClassifier(n_estimators=50, max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X[train_index], y[train_index])
    pred = rf.predict(X[val_index])
    acc, prec, rec, f1 = get_metrics(y[val_index], pred)
    print(f'Fold {i+1}: Accuracy = {acc:.4f} | F1 = {f1:.4f}')
    acc_rf_list.append(acc)
    f1_rf_list.append(f1)

acc_rf = np.mean(acc_rf_list)
f1s_rf = np.mean(f1_rf_list)
print(f'Mean CV Accuracy: {acc_rf*100:.2f}% ± {np.std(acc_rf_list)*100:.2f}% | Mean F1: {f1s_rf*100:.2f}%')


In [ ]:
# Multinomial Naive Bayes
acc_nb_list, f1_nb_list = [], []
print('--- Naive Bayes Classifier ---')
for i, (train_index, val_index) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y)):
    classifier = MultinomialNB(alpha=0.1)
    classifier.fit(X[train_index], y[train_index])
    pred = classifier.predict(X[val_index])
    acc, prec, rec, f1 = get_metrics(y[val_index], pred)
    print(f'Fold {i+1}: Accuracy = {acc:.4f} | F1 = {f1:.4f}')
    acc_nb_list.append(acc)
    f1_nb_list.append(f1)

acc_nb = np.mean(acc_nb_list)
f1s_nb = np.mean(f1_nb_list)
print(f'Mean CV Accuracy: {acc_nb*100:.2f}% ± {np.std(acc_nb_list)*100:.2f}% | Mean F1: {f1s_nb*100:.2f}%')


In [ ]:
# Comparison of ML models
plt.figure(figsize=(8, 5))
scores = ['Decision Tree', 'Random Forest', 'Naive Bayes']
accscore = [acc_dt, acc_rf, acc_nb]
f1score = [f1s_dt, f1s_rf, f1s_nb]
w = 0.35
bar1 = np.arange(len(scores))
bar2 = [i + w for i in bar1]

plt.bar(bar1, accscore, w, label='Accuracy', color='#3182ce')
plt.bar(bar2, f1score, w, label='F1 Score', color='#38a169')
plt.xticks(bar1 + w / 2, scores)
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.legend()
plt.title('Comparison Between ML Models (Human 5-Fold CV)')
plt.show()


### Train & Evaluate Final Model on Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

nb_ = MultinomialNB(alpha=0.1)
nb_.fit(X_train, y_train)

y_pred = nb_.predict(X_test)
acc, prec, rec, f1 = get_metrics(y_test, y_pred)
print(f'Test Accuracy: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}%')

# Save model
filename = 'finalized_model.sav'
pickle.dump(nb_, open(filename, 'wb'))
print(f'Saved model to {filename}')


In [ ]:
plt.figure(figsize=(7, 6))
cm_h = confusion_matrix(y_test, y_pred, labels=list(range(7)))
sns_plot = sns.heatmap(cm_h, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Confusion Matrix - Human Test Set (Naive Bayes)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
fig = sns_plot.get_figure()
fig.savefig('cm human.png', dpi=300, bbox_inches='tight')
plt.show()


### Zero-Shot Cross-Species Evaluation on Chimpanzee & Dog Data


In [ ]:
chimp = pd.read_table('chimp_data.txt')
dog = pd.read_table('dog_data.txt')
print('Chimp dataset shape:', chimp.shape)
print('Dog dataset shape:', dog.shape)


In [ ]:
chimp['words'] = chimp.apply(lambda x: getKmers(x['sequence']), axis=1)
dog['words'] = dog.apply(lambda x: getKmers(x['sequence']), axis=1)

chimp_texts = [' '.join(words) for words in chimp['words']]
y_c = chimp['class'].values

dog_texts = [' '.join(words) for words in dog['words']]
y_d = dog['class'].values

X_chimp = cv.transform(chimp_texts)
X_dog = cv.transform(dog_texts)
print('X_chimp shape:', X_chimp.shape)
print('X_dog shape:', X_dog.shape)


In [ ]:
# Predict on Chimpanzee data
y_pred_chimp = nb_.predict(X_chimp)
acc_c, prec_c, rec_c, f1_c = get_metrics(y_c, y_pred_chimp)
print('=== Performance on Chimpanzee Data ===')
print(f'Accuracy:  {acc_c*100:.2f}%')
print(f'Precision: {prec_c*100:.2f}%')
print(f'Recall:    {rec_c*100:.2f}%')
print(f'F1 Score:  {f1_c*100:.2f}%')

plt.figure(figsize=(7, 6))
cm_c = confusion_matrix(y_c, y_pred_chimp, labels=list(range(7)))
sns_plot_c = sns.heatmap(cm_c, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Confusion Matrix - Chimpanzee Data (Naive Bayes)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
fig_c = sns_plot_c.get_figure()
fig_c.savefig('cm chimp.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Predict on Dog data
y_pred_dog = nb_.predict(X_dog)
acc_d, prec_d, rec_d, f1_d = get_metrics(y_d, y_pred_dog)
print('=== Performance on Dog Data ===')
print(f'Accuracy:  {acc_d*100:.2f}%')
print(f'Precision: {prec_d*100:.2f}%')
print(f'Recall:    {rec_d*100:.2f}%')
print(f'F1 Score:  {f1_d*100:.2f}%')

plt.figure(figsize=(7, 6))
cm_d = confusion_matrix(y_d, y_pred_dog, labels=list(range(7)))
sns_plot_d = sns.heatmap(cm_d, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Confusion Matrix - Dog Data (Naive Bayes)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
fig_d = sns_plot_d.get_figure()
fig_d.savefig('cm dog.png', dpi=300, bbox_inches='tight')
plt.show()


# DNA Sequence Classification using Deep Learning (1D-CNN & Transformer)


In [ ]:
# Load raw sequences and labels from local files
from src.preprocessing import load_raw_sequences_and_labels

X_h_seq, y_h_seq, _ = load_raw_sequences_and_labels('human_sequence.txt', 'human_labels.txt', max_len=50)
X_c_seq, y_c_seq, _ = load_raw_sequences_and_labels('chimp_sequence.txt', 'chimp_labels.txt', max_len=50)
X_d_seq, y_d_seq, _ = load_raw_sequences_and_labels('dog_sequence.txt', 'dog_labels.txt', max_len=50)

print(f'Human: {X_h_seq.shape}, labels: {y_h_seq.shape}')
print(f'Chimp: {X_c_seq.shape}, labels: {y_c_seq.shape}')
print(f'Dog:   {X_d_seq.shape}, labels: {y_d_seq.shape}')


### 1. 1D Convolutional Neural Network (CNN)


In [ ]:
from src.models import DNA_CNN1D
from src.pipeline import train_and_evaluate_dl_model, evaluate_dl_model

X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_h_seq, y_h_seq, test_size=0.25, random_state=42, stratify=y_h_seq
)

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)

cnn_model = DNA_CNN1D(seq_len=50, in_channels=4, num_classes=7, num_filters=32, kernel_size=12, dropout=0.3)
cnn_hist, cnn_val_metrics = train_and_evaluate_dl_model(
    cnn_model, X_train_dl, y_train_dl, X_val_dl, y_val_dl, epochs=25, batch_size=32, lr=1e-3, device=device
)

print(f'CNN Human Validation Accuracy: {cnn_val_metrics["accuracy"]*100:.2f}% | F1: {cnn_val_metrics["f1"]*100:.2f}%')

# Plot Loss & Accuracy Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(cnn_hist['train_loss'], label='Train Loss')
ax1.plot(cnn_hist['val_loss'], label='Val Loss')
ax1.set_title('1D-CNN Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()

ax2.plot(cnn_hist['train_acc'], label='Train Acc')
ax2.plot(cnn_hist['val_acc'], label='Val Acc')
ax2.set_title('1D-CNN Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
# CNN Evaluation on Human, Chimp, and Dog
cm_cnn_h = cnn_val_metrics['confusion_matrix']
plt.figure(figsize=(7, 6))
sns.heatmap(cm_cnn_h, annot=True, fmt='d', cmap=plt.cm.Blues)
plt.title('CNN Normalized Confusion Matrix - Human')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.savefig('cnh cm.png', dpi=300, bbox_inches='tight')
plt.show()

chimp_cnn_metrics = evaluate_dl_model(cnn_model, X_c_seq, y_c_seq, device=device)
cm_cnn_c = chimp_cnn_metrics['confusion_matrix']
plt.figure(figsize=(7, 6))
sns.heatmap(cm_cnn_c, annot=True, fmt='d', cmap=plt.cm.Blues)
plt.title('CNN Confusion Matrix - Chimpanzee')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.savefig('cnc cm.png', dpi=300, bbox_inches='tight')
plt.show()

dog_cnn_metrics = evaluate_dl_model(cnn_model, X_d_seq, y_d_seq, device=device)
cm_cnn_d = dog_cnn_metrics['confusion_matrix']
plt.figure(figsize=(7, 6))
sns.heatmap(cm_cnn_d, annot=True, fmt='d', cmap=plt.cm.Blues)
plt.title('CNN Confusion Matrix - Dog')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.savefig('cnd cm.png', dpi=300, bbox_inches='tight')
plt.show()


### 2. DNA Transformer / Self-Attention Architecture


In [ ]:
from src.models import DNATransformer

trans_model = DNATransformer(seq_len=50, in_dim=4, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, num_classes=7, dropout=0.2)
trans_hist, trans_val_metrics = train_and_evaluate_dl_model(
    trans_model, X_train_dl, y_train_dl, X_val_dl, y_val_dl, epochs=25, batch_size=32, lr=1e-3, device=device
)

print(f'Transformer Human Validation Accuracy: {trans_val_metrics["accuracy"]*100:.2f}% | F1: {trans_val_metrics["f1"]*100:.2f}%')

# Transformer Cross-species evaluation
chimp_trans_metrics = evaluate_dl_model(trans_model, X_c_seq, y_c_seq, device=device)
dog_trans_metrics = evaluate_dl_model(trans_model, X_d_seq, y_d_seq, device=device)

print(f'Transformer Chimpanzee Accuracy: {chimp_trans_metrics["accuracy"]*100:.2f}%')
print(f'Transformer Dog Accuracy:        {dog_trans_metrics["accuracy"]*100:.2f}%')


### Comparison of All Algorithms


In [ ]:
models_list = ['Decision Tree', 'Random Forest', 'Naive Bayes', '1D-CNN', 'DNA Transformer']
acc_list = [acc_dt * 100, acc_rf * 100, acc_nb * 100, cnn_val_metrics['accuracy'] * 100, trans_val_metrics['accuracy'] * 100]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models_list, acc_list, color='#805ad5', width=0.45)
ax.set_xlabel('Model Architecture', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Comprehensive Model Accuracy Comparison (Human Dataset)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)

for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords='offset points',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()
